# 🚀 Модуль 11. Урок 3. Построение RAG-пайплайна (Retrieval-Augmented Generation) для базы знаний

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_11_rag_vector_db/lesson_03_building_rag_pipeline_analyst.ipynb)

## 📌 Бизнес-контекст и методология
RAG объединяет поиск по корпоративной базе знаний и генерацию ответа моделью:
1. Retrieval: поиск релевантных фрагментов документации.
2. Augmentation: добавление найденных фрагментов в контекст промпта.
3. Generation: генерация ответа со ссылками на источники.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
# Сборка RAG-промпта с цитированием контекста
retrieved_context = [
    "Документ #41: Формула расчета CAC = (Маркетинговые расходы + Зарплата сейлзов) / Новые клиенты.",
    "Документ #42: Бенчмарк окупаемости CAC для B2B SaaS составляет менее 12 месяцев."
]

user_query = "Как рассчитать CAC и каков здоровый срок окупаемости?"

rag_prompt = f"""
Используй ТОЛЬКО следующий контекст для ответа на вопрос:
{' '.join(retrieved_context)}

Вопрос: {user_query}
Ответь кратко и укажи номера использованных документов.
"""

print("Сформированный RAG-промпт:")
print(rag_prompt.strip())


Сформированный RAG-промпт:
Используй ТОЛЬКО следующий контекст для ответа на вопрос:
Документ #41: Формула расчета CAC = (Маркетинговые расходы + Зарплата сейлзов) / Новые клиенты. Документ #42: Бенчмарк окупаемости CAC для B2B SaaS составляет менее 12 месяцев.

Вопрос: Как рассчитать CAC и каков здоровый срок окупаемости?
Ответь кратко и укажи номера использованных документов.


### 🎯 Практическое задание 1: Проверка наличия контекста
Убедитесь, что оба документа переданы в RAG-промпт.

In [3]:
### ВАШ КОД ЗДЕСЬ
assert "Документ #41" in rag_prompt
assert "Документ #42" in rag_prompt
print("✅ Тест успешно пройден: контекстная инъекция выполнена корректно!")


✅ Тест успешно пройден: контекстная инъекция выполнена корректно!


### 💡 Борьба с галлюцинациями в RAG
Всегда добавляйте инструкцию: "Если в предоставленном контексте нет ответа, прямо ответь: 'Информация отсутствует в базе знаний'.".